# cli

> The CLI driver -- the correction core's first (and currently only) frontend.
> `run <decomp-manifest>` corrects the committed spine in the decomp graph DB,
> pointing the graph worker at that shared DB via load-time config, with optional
> session resume/reopen and a second-transcriber manifest for the cross-transcriber
> diff.

In [ ]:
#| default_exp cli

In [ ]:
#| export
import argparse
import asyncio
import logging
from pathlib import Path
from typing import Dict, List, Optional

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue

from cjm_transcript_correction_core.models import CorrectionConfig
from cjm_transcript_correction_core.pipeline import (
    run_correction, load_decomp_manifest, resolve_graph_db_path,
)

logger = logging.getLogger(__name__)

In [ ]:
#| export
def build_parser() -> argparse.ArgumentParser:  # Configured CLI parser
    """Build the CLI parser (subcommands: run)."""
    parser = argparse.ArgumentParser(
        prog="cjm-transcript-correction-core",
        description="Headless transcript correction: non-destructive overlay on a committed decomp spine.",
    )
    sub = parser.add_subparsers(dest="command", required=True)
    run = sub.add_parser("run", help="Correct a decomp-core run manifest's committed spine")
    run.add_argument("manifest", help="Decomp-core run manifest JSON (the committed spine to correct)")
    run.add_argument("--secondary-manifest", default=None,
                     help="Second-transcriber decomp manifest of the same source (cross-transcriber diff)")
    run.add_argument("--manifests-dir", default=".cjm/manifests", help="Capability manifests directory")
    run.add_argument("--graph-plugin", default="cjm-graph-plugin-sqlite", help="Graph-storage capability name")
    run.add_argument("--graph-db-path", default=None,
                     help="Override graph DB path (default: the decomp manifest's recorded db_path)")
    run.add_argument("--sysmon-plugin", default=None,
                     help="MonitorPlugin for empirical attribution (CR-7); loaded first; default: none")
    run.add_argument("--session", default=None, help="Resume an existing CorrectionSession id")
    run.add_argument("--reopen", action="store_true", help="Reopen a completed session (with --session)")
    run.add_argument("--actor", default="human", help="Actor recorded on corrections + review markers")
    run.add_argument("--no-prune", action="store_true", help="Skip the D14 empty-segment prune")
    run.add_argument("-y", "--yes", action="store_true", help="Auto-accept HITL seams (headless mode)")
    run.add_argument("--output", default=None, help="Correction-manifest output path (default: runs/<run_id>.json)")
    run.add_argument("-v", "--verbose", action="store_true", help="DEBUG-level logging")
    return parser

In [ ]:
#| export
def load_capabilities(
    manager: PluginManager,                      # Freshly constructed manager
    instance_ids: List[str],                     # Capability names to load, in order
    configs: Optional[Dict[str, Dict]] = None,   # Per-capability load-time config (e.g. graph db_path)
) -> None:
    """Discover manifests + load each capability, passing per-capability config (CR-2 caller-wins)."""
    configs = configs or {}
    manager.discover_manifests()
    discovered = {m.name: m for m in manager.discovered}
    for iid in instance_ids:
        meta = discovered.get(iid)
        if meta is None:
            raise SystemExit(
                f"capability {iid!r} not found in manifests "
                f"(discovered: {sorted(discovered)}) -- run cjm-ctl install-all first"
            )
        if not manager.load_plugin(meta, config=configs.get(iid)):
            raise SystemExit(f"failed to load capability {iid!r}")
        logger.info(f"loaded {iid}" + (f" (db_path override)" if iid in configs else ""))

In [ ]:
#| export
async def run_command(
    args: argparse.Namespace,  # Parsed args for the `run` subcommand
) -> int:  # Process exit code
    """Execute the `run` subcommand: correct a decomp manifest's committed spine."""
    manifest_path = str(Path(args.manifest).resolve())
    if not Path(manifest_path).exists():
        raise SystemExit(f"decomp manifest not found: {manifest_path}")

    decomp = load_decomp_manifest(manifest_path)
    graph_db_path = resolve_graph_db_path(decomp, args.graph_plugin, override=args.graph_db_path)
    if not graph_db_path:
        raise SystemExit("could not resolve graph DB path from manifest; pass --graph-db-path explicitly")

    cfg = CorrectionConfig(
        graph_plugin=args.graph_plugin, graph_db_path=graph_db_path,
        actor=args.actor, assume_yes=args.yes, prune_empty=not args.no_prune,
    )

    manager = PluginManager(
        search_paths=[Path(args.manifests_dir)],
        sysmon_plugin_name=args.sysmon_plugin,
    )
    load_order = ([args.sysmon_plugin] if args.sysmon_plugin else []) + [cfg.graph_plugin]
    # Point the graph worker at the decomp graph DB (the shared spine) via load-time config.
    load_capabilities(manager, load_order, configs={cfg.graph_plugin: {"db_path": graph_db_path}})

    queue = JobQueue(deps=manager, sysmon_plugin_name=args.sysmon_plugin)
    await queue.start()
    try:
        manifest = await run_correction(
            manager, queue, cfg, manifest_path, graph_db_path,
            session_id=args.session, reopen=args.reopen,
            secondary_manifest_path=(
                str(Path(args.secondary_manifest).resolve()) if args.secondary_manifest else None
            ),
        )
    finally:
        await queue.stop()
        for iid in reversed(load_order):
            try:
                manager.unload_plugin(iid)
            except Exception as e:  # Best-effort teardown; never mask the run's outcome
                logger.warning(f"unload {iid} failed: {e}")

    out = Path(args.output) if args.output else Path("runs") / f"{manifest.run_id}.json"
    manifest.save(out)
    n_docs = len(manifest.documents)
    n_pruned = sum(d.get("pruned", 0) for d in manifest.documents)
    n_flagged = sum(d.get("worklist_flagged", 0) for d in manifest.documents)
    print(f"correction manifest: {out}")
    print(f"documents: {n_docs}  worklist flagged: {n_flagged}  pruned: {n_pruned}")
    print(f"session: {manifest.session_id}")
    return 0

In [ ]:
#| export
def main(
    argv: Optional[List[str]] = None,  # Argument list override (None = sys.argv)
) -> int:  # Process exit code
    """CLI entry point (console script: `cjm-transcript-correction-core`)."""
    args = build_parser().parse_args(argv)
    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s :: %(message)s",
    )
    if args.command == "run":
        return asyncio.run(run_command(args))
    raise SystemExit(f"unknown command: {args.command}")

In [ ]:
#| eval: false
# CLI smoke check (parser only; no runtime)
p = build_parser()
ns = p.parse_args(["run", "/tmp/decomp.json", "--secondary-manifest", "/tmp/vox.json", "-y"])
assert ns.command == "run" and ns.yes and ns.secondary_manifest == "/tmp/vox.json"
print("cli parser checks OK")

cli parser checks OK
